In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
import cv2
from tqdm import tqdm
import argparse
import gdown
import matplotlib.pyplot as plt
import sys, os
#using hugging face models for classification and image to text(dinov2 and llava)
from transformers import AutoProcessor, VipLlavaForConditionalGeneration
from transformers import AutoImageProcessor, AutoModelForImageClassification
import warnings
warnings.filterwarnings("ignore")
import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding
import random
import shutil

#IMPORTANT GOOGLE COLAB USES cv2_imshow not cv2.imshow


Use line bellow only if you're running on google colab


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
%cd /content/drive/MyDrive/PerSense


/content/drive/MyDrive/PerSense


In [10]:
!ls

data		GroundingDINO  output3		     PerSense	  ref_images	    yolov5su.pt
DSALVANet	LICENSE        outputs		     persense.py  requirements.txt
eval_miou.py	output	       outputs_final	     __pycache__  show.py
groundedsam.py	output2        per_segment_anything  README.md	  ViPLLaVA


In [18]:
!pwd


/content/drive/MyDrive/PerSense


In [19]:
#!mkdir /content/drive/MyDrive/PerSense/data/Images

In [20]:
#!mkdir /content/drive/MyDrive/PerSense/data/Test

In [21]:
!pip install torchshow

In [22]:
!pip install -r requirements.txt

# removed numpy==1.21.2
#removed transformers


  Cloning https://github.com/facebookresearch/segment-anything.git to /tmp/pip-req-build-qbi8230t
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git /tmp/pip-req-build-qbi8230t
  Resolved https://github.com/facebookresearch/segment-anything.git to commit 6fdee8f2727f4506cfbbe553e23b895e27956588
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.3/82.3 kB 9.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.6 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu1

In [23]:
from show import *
from google.colab.patches import cv2_imshow
from per_segment_anything import sam_model_registry, SamPredictor
#from content.DSALVANet.utils.PerSense_modules.py import IDM
#from DSALVANet.utils.PerSense_modules import IDM
#from DSALVANet.utils.data_preprocess import preprocess
#from DSALVANet.utils.model_helper import build_model

from PIL import Image , ImageChops
from torchvision import transforms

from GroundingDINO.groundingdino.util.inference import Model
from typing import List
import supervision as sv
import gc
import time
import torchshow


In [24]:
print("======> Load SAM" ) # SAM small !
sam_type, sam_ckpt = 'vit_b', 'data/sam_vit_b_01ec64.pth'
sam = sam_model_registry[sam_type](checkpoint=sam_ckpt).cuda()
print("======> Done" )


======> Load SAM
======> Done


In [25]:
print("======> Load Grounding Detector" )
GROUNDING_DINO_CONFIG_PATH = "GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py"
GROUNDING_DINO_CHECKPOINT_PATH = "GroundingDINO/weights/groundingdino_swint_ogc.pth"
grounding_dino_model = Model(model_config_path=GROUNDING_DINO_CONFIG_PATH, model_checkpoint_path=GROUNDING_DINO_CHECKPOINT_PATH)
print("======> Done" )

======> Load Grounding Detector
final text_encoder_type: bert-base-uncased


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

======> Done


In [26]:
"""
print("======> Load Object Counter" )
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# directly set values !
weight_path = "./DSALVANet/checkpoints/checkpoint_200.pth"  #
visualize = False

#
# parser_input = argparse.ArgumentParser(description="Test code of DSALVANet")
# parser_input.add_argument("-w", "--weight", type=str, default="./DSALVANet/checkpoints/checkpoint_200.pth", help="Path of weight.")
# parser_input.add_argument('--visualize', type=bool, default= False)
# args_counter = parser_input.parse_args()
# weight_path = args_counter.weight

counter_model = build_model(weight_path, device)
print("======> Done" )
"""

'\nprint("======> Load Object Counter" )\ndevice = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n\n# directly set values !\nweight_path = "./DSALVANet/checkpoints/checkpoint_200.pth"  #\nvisualize = False\n\n#\n# parser_input = argparse.ArgumentParser(description="Test code of DSALVANet")\n# parser_input.add_argument("-w", "--weight", type=str, default="./DSALVANet/checkpoints/checkpoint_200.pth", help="Path of weight.")\n# parser_input.add_argument(\'--visualize\', type=bool, default= False)\n# args_counter = parser_input.parse_args()\n# weight_path = args_counter.weight\n\ncounter_model = build_model(weight_path, device)\nprint("======> Done" )\n'

In [27]:


print("=========>Loading Image classifier")
processor_classifier  =  AutoImageProcessor.from_pretrained('facebook/dinov2-small-imagenet1k-1-layer')
model_classifier = AutoModelForImageClassification.from_pretrained('facebook/dinov2-small-imagenet1k-1-layer')


print("========> Done")


=========>Loading Image classifier


preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/58.3k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/91.3M [00:00<?, ?B/s]

========> Done


In [28]:
print("=========>Loading LLavA")
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_lava = VipLlavaForConditionalGeneration.from_pretrained("llava-hf/vip-llava-7b-hf", device_map="auto", torch_dtype=torch.float16)
processor_lava = AutoProcessor.from_pretrained("llava-hf/vip-llava-7b-hf")
print("=========> Done")



=========>Loading LLavA


config.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/70.3k [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/41.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


=========> Done


In [29]:
images_path = '/content/drive/MyDrive/PerSense/data/Images'
masks_path = '/content/drive/MyDrive/PerSense/data/Images'
output_path = '/content/drive/MyDrive/PerSense/output'
test_path = '/content/drive/MyDrive/PerSense/data/Test'
template_image_path = '/content/drive/MyDrive/PerSense/data/Images/01.png' #image for classification used in dino and llava




In [30]:
print("=========>Predicting Image class with classifier")
template_image_path = '/content/drive/MyDrive/PerSense/data/Images/01.png'
template_image =  Image.open(template_image_path)
inputs =  processor_classifier(images = template_image, return_tensors = 'pt')
outputs =  model_classifier(**inputs)
logits = outputs.logits
predicted_class_idx = logits.argmax(-1).item()
print("Predicted class:", model_classifier.config.id2label[predicted_class_idx])
class_name = model_classifier.config.id2label[predicted_class_idx]
print("=========> Done")

=========>Predicting Image class with classifier
Predicted class: carton
=========> Done


In [31]:
print("========>Prompting llava with class name")
prompt = "A chat between a curious human and an artificial intelligence assistant that detects objects on pallet. The assistant gives helpful, detailed, and polite answers to the human's questions.###Human: <image>\n{}###Assistant:"
question = "Name the " + class_name + "object in the image ?"
prompt =  prompt.format(question)
inputs_llava = processor_lava(text=prompt, images=template_image, return_tensors="pt").to(0, torch.float16)
generate_ids = model_lava.generate(**inputs_llava, max_new_tokens=20)
processor_lava.decode(generate_ids[0][len(inputs_llava["input_ids"][0]):], skip_special_tokens=True)
sentence = processor_lava.decode(generate_ids[0][len(inputs_llava["input_ids"][0]):], skip_special_tokens=True)

sentence =  sentence.replace(".", "")
sentence =sentence.split()
last_word = sentence[-1]
class1 = last_word
print(class1)
print("=============>Done")

========>Prompting llava with class name


We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


box
=============>Done


In [32]:
print("==============> Releasing Models from memory ")
del model_classifier
del model_lava
del processor_classifier
del processor_lava
del template_image
gc.collect()
torch.cuda.empty_cache()

print("=============> Done")

==============> Releasing Models from memory 
=============> Done


In [33]:

#optional
def enhance_class_name(class_names: List[str]) -> List[str]:
    vowels = {'a', 'i', 'o', 'u'}
    enhanced_names = []
    for class_name in class_names:
        # Check if the word ends with 's' or 'y'
        if class_name[-1].lower() == 's':
            enhanced_names.append(f"all {class_name}")
        elif class_name[-1].lower() == 'y':
            if len(class_name) > 1 and class_name[-2].lower() == 'e':
                enhanced_names.append(f"all {class_name}s")
            else:
                enhanced_names.append(f"all {class_name[:-1]}ies")
         # Check if the word ends with 'h'
        elif class_name[-1].lower() == 'h':
            enhanced_names.append(f"all {class_name}es")
        elif class_name[-1].lower() == 'x': # added 'x' exception ! -> made it into not boxes
            enhanced_names.append(f"not {class_name}es")
        # For all other cases
        elif class_name[-1].lower() in vowels:
            enhanced_names.append(f"all {class_name}es")
        else:
            enhanced_names.append(f"all {class_name}s")
    return enhanced_names



In [34]:
#optional only if DSALVA IS USED
def point_prompt_select(similarity, point_priors, cnt, dino_boxes,buffer = 5):
    max_conf = torch.tensor(similarity.max()),
    # b = similarity.min()

    final_pts = []
    for pnt in point_priors:
        check = similarity [pnt[1], pnt[0]] # coords inverted because it is coming in this format from the counter
        if cnt == 1:
            a = torch.sqrt(torch.tensor([4]))
        else:
            a = torch.div(torch.tensor([cnt]), torch.sqrt(torch.tensor([2])))
        threshold = torch.div(max_conf[0],a.cuda())
        verified_point = False
        for boxes in dino_boxes.xyxy:
            if (check >= threshold) and \
               (boxes[0] + buffer <= pnt[0] <= boxes[2] - buffer) and \
               (boxes[1] + buffer <= pnt[1] <= boxes[3] - buffer): # To check if the point lies within the bounding boxes predicted by DINO
                verified_point = True
                break
        if verified_point:
            final_pts.append(pnt)


    return final_pts

In [35]:
#Unused
def point_selection(mask_sim, topk=1):
    # Top-1 point selection
    w, h = mask_sim.shape
    topk_xy = mask_sim.flatten(0).topk(topk)[1]
    topk_x = (topk_xy // h).unsqueeze(0)
    topk_y = (topk_xy - topk_x * h)
    topk_xy = torch.cat((topk_y, topk_x), dim=0).permute(1, 0)
    topk_label = np.array([1] * topk)
    topk_xy = topk_xy.cpu().numpy()

    return topk_xy, topk_label

In [36]:
def release_memory(test_image, test_feat):
    # Clear variables
    del test_image, test_feat
    # Explicitly release GPU memory
    torch.cuda.empty_cache()
    # Perform garbage collection
    gc.collect()

In [37]:
#\
def filtered_similarity(sim_matrix, bbox):
    cimg = np.zeros_like(sim_matrix.cpu(), np.uint8)
    bbox = np.int32(bbox)
    cimg_box = cv2.rectangle(cimg,(bbox[0],bbox[1]), (bbox[2], bbox[3]) , 255 , -1)
    center_pt = ((bbox[0]+ bbox[2])//2, (bbox[1]+bbox[3])//2)
    center_pt = np.int32(center_pt)
    radius = 2
    cv2.circle(cimg_box, center_pt, radius, (255, 255, 0), 2)
    sim_mat_new = np.multiply(sim_matrix.cpu(), cimg_box)

    return sim_mat_new, center_pt

In [41]:
def overlay(image, mask, color, alpha, resize=None):



    color = color[::-1]
    colored_mask = np.expand_dims(mask, 0).repeat(3, axis=0)
    colored_mask = np.moveaxis(colored_mask, 0, -1)
    masked = np.ma.MaskedArray(image, mask=colored_mask, fill_value=color)
    image_overlay = masked.filled()

    if resize is not None:
        image = cv2.resize(image.transpose(1, 2, 0), resize)
        image_overlay = cv2.resize(image_overlay.transpose(1, 2, 0), resize)

    image_combined = cv2.addWeighted(image, 1 - alpha, image_overlay, alpha, 0)

    return image_combined

In [71]:


def visualization(test_image, masks, output_path, test_idx):

    masks_np  =  masks.cpu().detach().numpy()

    masked = np.zeros_like(masks_np[0], dtype=np.uint8)
    for mask in masks_np:

      masked = cv2.bitwise_or(masked, mask.astype(np.uint8))
    #fig = plt.figure(figsize=(10, 10))

    #plt.imshow(test_image)

    # Convert image to the format for cv2 and mask overlay
    image_with_masks = test_image.copy()


    for idx, mask_i in enumerate(masks):
        mask_i = mask_i.cpu().detach().numpy().astype(np.uint8)
        color = [random.randrange(255) for _ in range(3)]
        random.shuffle(color)  # Shuffle color to get random colors

        # Overlay mask on image
        image_with_masks = overlay(image_with_masks, mask_i, color, alpha=0.6)

    # Display the image with masks
    cv2_imshow(image_with_masks)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

    # Save the output image
    vis_mask_output_path = os.path.join(output_path, f'vis_mask_{test_idx}.jpg')
    cv2.imwrite(vis_mask_output_path, image_with_masks)
    print(f"Masked image saved to: {vis_mask_output_path}")

In [65]:

def remove_ipynb_checkpoints(dir_path):

    checkpoints_path = os.path.join(dir_path, '.ipynb_checkpoints')

    if os.path.exists(checkpoints_path) and os.path.isdir(checkpoints_path):
        shutil.rmtree(checkpoints_path)
        print(f"Removed: {checkpoints_path}")
    else:
        print(f"No .ipynb_checkpoints directory found in {dir_path}")

remove_ipynb_checkpoints(test_path)
remove_ipynb_checkpoints(images_path)


No .ipynb_checkpoints directory found in /content/drive/MyDrive/PerSense/data/Test
No .ipynb_checkpoints directory found in /content/drive/MyDrive/PerSense/data/Images


In [40]:

class Mask_Weights(nn.Module):
    def __init__(self):
        super().__init__()
        self.weights = nn.Parameter(torch.ones(2, 1, requires_grad=True) / 3)

In [75]:
#implementing persense function

import matplotlib.pyplot as plt



def persense(obj_name, images_path, masks_path, output_path, sam, grounding_dino_model, infer_time,class_name , visualize):
    obj_count = 0
    avg_iter = 0
    ref_idx = '00'

    print("\n ----------> Segment " +  obj_name)

    ref_image_path = os.path.join(images_path, obj_name[:2] + '.png')
    ref_mask_path = os.path.join(masks_path, obj_name[:2] + '.jpg')

    print("Reference image path:", ref_image_path)
    print("Reference mask path:", ref_mask_path)
  #  test_images_path = []
  #  for obj in os.listdir(test_path):
  #    test_images_path.append(os.path.join(test_path, obj + '.png'))
  #    print(os.path.join(test_path, obj[:2] + '.png'))



#    test_images_path = os.path.join(test_path, obj_name[:2] + '.png')
    output_obj_path = os.path.join(output_path, obj_name)

    if not os.path.exists(output_obj_path):
        os.makedirs(output_obj_path)

    # Check if reference image and mask exist
    if not os.path.exists(ref_image_path) or not os.path.exists(ref_mask_path):
        print(f"Reference image or mask not found for {obj_name}. Skipping.")
        return

    ref_image = cv2.imread(ref_image_path)
    if ref_image is None:
        print(f"Failed to load reference image at {ref_image_path}. Skipping.")
        return

    ref_image = cv2.cvtColor(ref_image, cv2.COLOR_BGR2RGB)

    ref_mask  =  cv2.imread(ref_mask_path)
    if ref_mask is None:
        print(f"Failed to load reference mask at {ref_mask_path}. Skipping.")
        return

    #tensor
    gt_mask = torch.tensor(ref_mask)[:,:,0]>0
    gt_mask =  gt_mask.float().unsqueeze(0).flatten(1).cuda()
    print("this is the shape of gt_mask", gt_mask.shape)



    for img_name in os.listdir(images_path):
        if ".DS" not in img_name:
            img_path = os.path.join(images_path, img_name)
            if os.path.isfile(img_path):
                print(f"Processing {img_path}")

                image = cv2.imread(img_path)
                if image is not None:
                    pass
                else:
                    print(f"Failed to load image at {img_path}.")
            else:
                print(f"Skipping non-file {img_path}")
    device  = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
    supp_mask =  Image.open(ref_image_path).convert("RGB")
    supp_image =  Image.open(ref_mask_path).convert("RGB")
    raw_image =  ImageChops.multiply(supp_image, supp_mask)
    #cv2_imshow(raw_image)
    raw_image_np = np.array(raw_image)
  #  cv2_imshow(raw_image_np)
    raw_image.save("./ref_images/masked_img.png", "PNG")
    image_file = "./ref_images/masked_img.png"


    for name, param in sam.named_parameters():
      param.requirements_grad = False
    predictor = SamPredictor(sam)

    print("====>Obtain self location prio")
    #===========================================image features encoding=====================================================================
  #  print("old ref_mas shape :  ",ref_mask.shape)
    ref_mask =  predictor.set_image(ref_image,ref_mask)
 #   print("new ref mask shape :" , ref_mask.shape)
    ref_feat = predictor.features.squeeze().permute(1, 2, 0)
#print("ref_feat shape : ",ref_feat.shape)

    ref_mask = F.interpolate(ref_mask, size=ref_feat.shape[0: 2], mode="bilinear")
    ref_mask = ref_mask.squeeze()[0]
 #   print("ref mask shape after interpolation : ", ref_mask.shape)

    #Target feature extraction
    target_feat  = ref_feat[ref_mask > 0]
    target_embedings = target_feat.mean(0).unsqueeze(0)
    print("target_embedings shape : ",target_embedings.shape)
    target_feat =  target_embedings/target_embedings.norm(dim=-1, keepdim=True)
    print("target_feat shape after operation  : ",target_feat.shape)
    target_embedings = target_embedings.unsqueeze(0)

    print('===> Testing Start')
    loop_over =  len(os.listdir(test_path))
    for test_idx  in tqdm(range(loop_over)):

      #load  test image
      #print(os.listdir(test_path)[test_idx])
      test_image_path = os.path.join(test_path, os.listdir(test_path)[test_idx])
      test_image = cv2.imread(test_image_path)
      test_image = cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB)
    #  cv2_imshow(test_image)

      predictor.set_image(test_image)
      test_feat =  predictor.features.squeeze()
      print('test_feat shape : ', test_feat.shape)
      #cosine similarity for attention map , currently not used (floor ? )
      C, h, w,  = test_feat.shape
      test_feat = test_feat / test_feat.norm(dim=0, keepdim=True)
      test_feat = test_feat.reshape(C, h * w)
      sim = target_feat @ test_feat

      sim = sim.reshape(1, 1, h, w)
      sim = F.interpolate(sim, scale_factor=4, mode="bilinear")
      sim = predictor.model.postprocess_masks(
                        sim,
                        input_size=predictor.input_size,
                        original_size=predictor.original_size).squeeze()
      print("=====> Running Persense")

      SOURCE_IMAGE_PATH = test_image_path


      #CLASSES = [class1[0], class2[0]] ,
      CLASSES  =   [class_name]
      #class2 = enhance_class_name(class_names = class1)

#=========== BEST THRESHOLD FOR BOX DETECTION IS 0.41==================================================
      BOX_THRESHOLD = 0.30
     # BOX_THESHOLD2 = 0.20

      TEXT_THRESHOLD = 0.10

      #load image
      image  = cv2.imread(SOURCE_IMAGE_PATH)

      detections =  grounding_dino_model.predict_with_classes(
          image = image,
          # classes=enhance_class_name(class_names=CLASSES)
          classes = CLASSES,
          box_threshold=BOX_THRESHOLD,
          text_threshold=TEXT_THRESHOLD,
      )
#=========================BOX FILTERING OPTIONAL ==========================
      areas = detections.box_area
      mean_area = np.mean(areas)
      std_area =  np.std(areas)
      # threshold for filtering example 2 stds above mean ()
      threshold = mean_area + 2 * std_area
      detections = detections[detections.box_area <= threshold]

#=====================================================================================
      #annotate image with detections

      class_conf  = detections.confidence
      index_conf =  np.argmax(class_conf)
      bbox_coord =  detections.xyxy[index_conf]

      annotated_image =  image.copy()
      for i in range(len(detections)):
        bbox =  detections.xyxy[i]
        confidence = detections.confidence[i]

        #draw bounding box
        cv2.rectangle(annotated_image,(int(bbox[0]),int(bbox[1])),(int(bbox[2]),int(bbox[3])),(0,255,0),2)

        label =  f"{'box'}: {confidence:.2f}"
        cv2.putText(annotated_image, label, (int(bbox[0]), int(bbox[1] - 10)), cv2.FONT_HERSHEY_SIMPLEX,0.5, (0, 255, 0), 2)

        annotated_image_rgb =  cv2.cvtColor(annotated_image,cv2.COLOR_BGR2RGB)
      if visualize ==True:

        plt.figure(figsize=(10, 10))
        plt.imshow(annotated_image_rgb)
        plt.axis('off')
        plt.show()









      #Positive location prior
      top_list = []
      filt_sim ,  cntr_pt  = filtered_similarity(sim,bbox_coord) # filtering  the values of cosine similarity falling under BBox with maximum conf value
      topk_xy_NA , topk_label =  point_selection(filt_sim,topk=1)

      top_list.append(cntr_pt)
      topk_xy =  top_list[0]
      topk_xy = np.array(top_list)

      #obtain target guidance for cross attention layers
      sim_tgt = (sim - sim.mean())/torch.std(sim)
      sim_tgt =  F.interpolate(sim_tgt.unsqueeze(0).unsqueeze(0),size = (64,64),mode="bilinear")
      attn_sim =  sim_tgt.sigmoid_().unsqueeze(0).flatten(3)

      print("shape of sim_tgt", sim_tgt.shape)
      print("shape of attn_sim", attn_sim.shape)

     # CLASSES =  class2
      print(CLASSES)


      release_memory(test_image, test_feat )
      input_boxes = torch.tensor([detections.xyxy], device= predictor.device)
      transformed_boxes = predictor.transform.apply_boxes_torch(input_boxes, test_image.shape[:2])







      #first step predtiction

      masks, scores, logits, _ = predictor.predict_torch(
          point_coords = None,
          point_labels = None,
          multimask_output=False,
          target_embedding = target_embedings,
          attn_sim =None,#attn_sim,
          boxes = transformed_boxes,

          )
      print("Masks tensor shape" , masks.shape)
      masks_np  =  masks.cpu().detach().numpy()
      print("Masks np array shape ", masks_np.shape)
      print("Test image shape ", test_image.shape)

      height,width,_ = test_image.shape
      masked = np.zeros_like(masks_np[0], dtype=np.uint8)
      image_copy = test_image.copy()
      for mask in masks_np:
        masked = cv2.bitwise_or(masked, mask.astype(np.uint8))

      image_with_masks = np.copy(test_image)
      channels = [0, 255]

      release_memory(test_image, test_feat )
      #================== CASCADED POST REFINEMENT ===========================



      masks , scores, logits,_ =  predictor.predict_torch(
          point_coords = None,
          point_labels = None,
          multimask_output = False,
          target_embedding = target_embedings,

          mask_input = logits,

          boxes = transformed_boxes,
          #attn_sim = attn_sim,
      )


      #============== CASCADED POST REFINEMENT 2 =================================
      masks, scores, logits,_ =  predictor.predict_torch(
          point_coords = None,
          point_labels = None,
          multimask_output = False,
          target_embedding = target_embedings,
          mask_input = logits,
          boxes = transformed_boxes)


      #================= Visalize =======================================================
      if visualize == True:
        visualization(test_image, masks, output_path, test_idx)















#=============== Part of code  that contains previous cascaded post refinement and usage of DSALVA =========================================================
"""

      best_idx =  np.argmax(scores_np)

      #cascaded post refinement 1
      masks , scores , logits, _ =  predictor.predict(
          point_coords =  topk_xy,
          point_labels = topk_label,
          mask_input =  logits_np[best_idx: best_idx +1, :, :],
          multimask_output = True
      )
      best_idx = np.argmax(scores)

      #cascaded post_refinement 2
      y,x = np.nonzero(masks[best_idx])
      x_min  = x.min()
      y_min = y.min()
      x_max = x.max()
      y_max = y.max()
      input_box = np.array([x_min,y_min,x_max,y_max])

      masks , scores , logits, _ = predictor.predict(
          point_coords =  topk_xy,
          point_labels = topk_label,
          box = input_box[None,:],
          mask_input = logits[best_idx: best_idx +1, :, :],
          multimask_output = True
      )
      best_idx = np.argmax(scores)
#      print("best_idx shape - > ", best_idx.shape)
#



      #write coordinates of bboxes
      lines  = [str(y_min), str(x_min), str(y_max), str(x_max),str(scores[best_idx])]
      with open('./DSALVANet/test_data/bbox.txt', 'w') as f:

        for line in lines:
          f.write(line)
          f.write(' ')
      #change here , not argparse but directly pass the args ?
      #description = ""
      #parser = argparse.ArgumentParser(description="Test code of DSALVANet")
      # parser.add_argument("-w", "--weight", type=str, default="/home/muhammad.siddiqui/Desktop/muhammad.siddiqui/Personalize-SAM/DSALVANet/checkpoints/checkpoint_200.pth", help="Path of weight.")
      #parser.add_argument("-i", "--img", type=str, default= test_image_path, help="Path of query image.")
      #parser.add_argument("-b", "--boxes", type=str, default="./DSALVANet/test_data/bbox.txt", help="Path of bbox coord txt file ")
      #parser.add_argument('--visualize', type=bool, default= False)
      description = "Test code of DSALVANet"
      boxes = "./DSALVANet/test_data/bbox.txt"
      img_path , boxes_path = test_image_path,boxes
      with open(boxes_path, 'r') as f:
        lines  = f.readlines()
        ori_boxes = []
        for line in lines:
          data = line.split()
          ori_boxes.append(list(map(int,data[0:4])))
      src_img =  cv2.imread(img_path)
      query,supports = preprocess(src_img, ori_boxes,device)
      output =  counter_model(query,supports)
      print("output shape : ",output.shape)
      print("Counter model run succesful")

    #check if this is to be changed from detections2 to detections for better results
      vis_output, pt_priors, count = IDM(src_img,ori_boxes,output, test_idx)
      max_conf_pt = topk_xy[0] # including the max conf point in the prompt list
      pt_priors = point_prompt_select(sim, pt_priors, count, detections) # to compare possible points with similrity map for filtering the accurate ones

      pt_list = []

      cnt = 0
      for pt in pt_priors:
        pt =  pt.cpu().detach().numpy().astype(np.int64)

        cnt += 1
        if cnt == 1 :
          pt_list.append(pt)
        pt_list[0]  = pt
        point  =  np.array(pt_list)
        print("point shape : ",point.shape)

        masks, scores, logits , _ =  predictor.predict(
            point_coords =  point,
            point_labels =  topk_label,
            multimask_output = True,
            attn_sim = attn_sim, #target guided attention
            target_embedding  = target_embedings, #target sematic prompting
        )
        best_idx = np.argmax(scores)

        masks,scores, logits, _ = predictor.predict(
            point_coords =  point,
            point_labels = topk_label,
            mask_input = logits[best_idx: best_idx +1, :, :],
            multimask_output = True)
        best_idx = np.argmax(scores)

        #cascaded post refinedment 2
        y,x =  np.nonzero(masks[best_idx])
        x_min  = x.min()
        y_min = y.min()
        x_max = x.max()
        y_max = y.max()
        input_box =  np.array([x_min,y_min,x_max,y_max])

        lines = [str(y_min), str(x_min), str(y_max), str(x_max), str(scores[best_idx])]
        with open('./DSALVANet/test_data/bbox.txt', 'a') as f:
                f.write('\n')
                for line in lines:
                    f.write(line)
                    f.write(' ')

        with open('./DSALVANet/test_data/bbox.txt', 'r') as file:
            lines = file.readlines()


        max_conf_bbox = lines.pop(0)
        print(lines)
        valid_lines = [line for line in lines if len(line.split()) > 4]
        sorted_lines = sorted(valid_lines, key=lambda x: float(x.split()[4]), reverse=True)
        #sorted_lines = sorted(lines, key=lambda x: float(x.split()[4]) if len(x.split()) > 4 else 0, reverse=True)


        with open('./DSALVANet/test_data/bbox.txt', 'w') as file:
            file.write(max_conf_bbox)  # Write back the header
            file.writelines(sorted_lines)


        with open('./DSALVANet/test_data/bbox.txt', 'r') as file:
            lines = file.readlines()

        print(len(lines))
        first_n_lines = lines[:5]  #here ??

        with open('./DSALVANet/test_data/bbox.txt', 'w') as file:
            file.writelines(first_n_lines)
        img_path =  test_image_path
        boxes_path = "./DSALVANet/test_data/bbox.txt"
        with open(boxes_path, "r") as f:
                lines = f.readlines()
                ori_boxes = []
                for line in lines:
                    data = line.split()
                    ori_boxes.append(list(map(int,data[0:4])))
        src_img = cv2.imread(img_path)
        query, supports = preprocess(src_img, ori_boxes,device)
            # model = build_model(weight_path,device)
        output = counter_model(query,supports)

        vis_output, pt_priors, count = IDM(src_img,ori_boxes,output, test_idx)
        pt_priors_all = pt_priors
  # it was detections2  and changed to detections to check if the points generated are more precise ??
        pt_priors = point_prompt_select(sim, pt_priors, count, detections) # to compare possible points with similrity map for filtering the accurate ones
        if not pt_priors:
          pt_priors = pt_priors_all

        mask_list_final = []
        pt_list_final = []
        prompt_list_final = []
        cnt  = 0
        for pt in pt_priors:
          pt = pt.cpu().detach().numpy().astype(np.int64)
          cnt +=1
          if cnt == 1:
            pt_list_final.append(pt)
          pt_list_final[0] = pt
          point =  np.array(pt_list_final)

          masks,scores,logits,logits_high = predictor.predict(
              point_coords = point,
              point_labels = topk_label,
              multimask_output = True,
               )
          best_idz = np.argmax(scores)
          prompt_list_final.append(point)
          mask_list_final.append(masks[best_idz])
        best_idx = np.argmax(mask_list_final)

        visualization(test_image, mask_list_final, prompt_list_final, topk_label, output_path, test_idx)
        composite_mask = np.zeros_like(mask_list_final[0], dtype=np.uint8)  # Initialize composite mask
        for mask in mask_list_final:
            composite_mask |= mask.astype(np.uint8) * 255  # Combine all masks using logical OR operation
        mask_output_path = os.path.join(output_path, f'{test_idx}.png')
        cv2.imwrite(mask_output_path, composite_mask)


        release_memory(test_image, test_feat )







      #args_dsalva = parser.parse_args()
      #print(args_dsalva)






























"""












'\n\n      best_idx =  np.argmax(scores_np)\n\n      #cascaded post refinement 1\n      masks , scores , logits, _ =  predictor.predict(\n          point_coords =  topk_xy,\n          point_labels = topk_label,\n          mask_input =  logits_np[best_idx: best_idx +1, :, :],\n          multimask_output = True\n      )\n      best_idx = np.argmax(scores)\n\n      #cascaded post_refinement 2\n      y,x = np.nonzero(masks[best_idx])\n      x_min  = x.min()\n      y_min = y.min()\n      x_max = x.max()\n      y_max = y.max()\n      input_box = np.array([x_min,y_min,x_max,y_max])\n\n      masks , scores , logits, _ = predictor.predict(\n          point_coords =  topk_xy,\n          point_labels = topk_label,\n          box = input_box[None,:],\n          mask_input = logits[best_idx: best_idx +1, :, :],\n          multimask_output = True\n      )\n      best_idx = np.argmax(scores)\n#      print("best_idx shape - > ", best_idx.shape)\n#\n\n\n\n      #write coordinates of bboxes\n      lines

In [77]:
#if DSALVA IS TO BE USED DEFINE counter_model as function param

for obj_name in os.listdir(images_path):

        infer_time = 0
        if ".DS" not in obj_name:

          persense(obj_name, images_path, masks_path, output_path, sam, grounding_dino_model, infer_time,class1,True)
          break










 ----------> Segment 01.png
Reference image path: /content/drive/MyDrive/PerSense/data/Images/01.png
Reference mask path: /content/drive/MyDrive/PerSense/data/Images/01.jpg
this is the shape of gt_mask torch.Size([1, 319014])
Processing /content/drive/MyDrive/PerSense/data/Images/01.png
Processing /content/drive/MyDrive/PerSense/data/Images/01.jpg
====>Obtain self location prio
target_embedings shape :  torch.Size([1, 256])
target_feat shape after operation  :  torch.Size([1, 256])
===> Testing Start


  0%|          | 0/6 [00:00<?, ?it/s]

test_feat shape :  torch.Size([256, 64, 64])
=====> Running Persense
shape of sim_tgt torch.Size([1, 1, 64, 64])
shape of attn_sim torch.Size([1, 1, 1, 4096])
['box']
Masks tensor shape torch.Size([5, 1, 1080, 1920])
Masks np array shape  (5, 1, 1080, 1920)
Test image shape  (1080, 1920, 3)


 17%|█▋        | 1/6 [00:01<00:06,  1.37s/it]

test_feat shape :  torch.Size([256, 64, 64])
=====> Running Persense
shape of sim_tgt torch.Size([1, 1, 64, 64])
shape of attn_sim torch.Size([1, 1, 1, 4096])
['box']
Masks tensor shape torch.Size([7, 1, 1080, 1920])
Masks np array shape  (7, 1, 1080, 1920)
Test image shape  (1080, 1920, 3)


...

 67%|██████▋   | 4/6 [00:05<00:02,  1.37s/it]

test_feat shape :  torch.Size([256, 64, 64])
=====> Running Persense
shape of sim_tgt torch.Size([1, 1, 64, 64])
shape of attn_sim torch.Size([1, 1, 1, 4096])
['box']
Masks tensor shape torch.Size([7, 1, 1080, 1920])
Masks np array shape  (7, 1, 1080, 1920)
Test image shape  (1080, 1920, 3)


 83%|████████▎ | 5/6 [00:06<00:01,  1.37s/it]

test_feat shape :  torch.Size([256, 64, 64])
=====> Running Persense
shape of sim_tgt torch.Size([1, 1, 64, 64])
shape of attn_sim torch.Size([1, 1, 1, 4096])
['box']
Masks tensor shape torch.Size([13, 1, 1080, 1920])
Masks np array shape  (13, 1, 1080, 1920)
Test image shape  (1080, 1920, 3)


100%|██████████| 6/6 [00:08<00:00,  1.38s/it]


In [ ]:


gc.collect()
torch.cuda.empty_cache()
